# GRANAP for developers, 2: extend the component

**What you will learn:** add a state variable and a process to `GranapAnatomy` by subclassing it, run the new
component in a plant model, find it in the declarations and the schedule, and test it.

**You need:** developer tutorial 1 (the component formalism). **Run time:** about 10 seconds.

## 1. A new variable and its process

Suppose a model reading the anatomies needs each cell's **equivalent diameter**, the diameter of a disc of its area.
A subclass declares it as a state variable and computes it in a step:

- `state_variable` declares it: its unit, its description, its initial value, its scale (`Compartment`, the cells,
  walls and junctions of the sections) and `location="node"` (a value per node of the DataStructure);
- the method `_equivalent_diameter` computes it. Its argument, `cell_area`, names the variable it reads (one of
  `GranapAnatomy`'s), and the dictionary it returns, the variable it writes. `@state` places it in the schedule,
  after the anatomies are generated;
- `@dataclass` is needed on the subclass, as on every component.

In [1]:
from dataclasses import dataclass

import numpy as np

from openalea.granap.metafspm.component import GranapAnatomy
from openalea.metafspm.coupling.component import state_variable
from openalea.metafspm.data_structure.configs import ScalesConfig
from openalea.metafspm.solve.decorator import state


@dataclass
class GranapAnatomyWithDiameter(GranapAnatomy):
    equivalent_diameter: float = state_variable(unit="µm", description="the protoplast's equivalent diameter (of a "
                                                                       "disc of its area; 0: apoplast)",
                                                initialize=0., scale=ScalesConfig.Compartment, location="node",
                                                state_variable_type="NonInertialIntensive")

    @state
    def _equivalent_diameter(self, cell_area) -> dict:
        return {"equivalent_diameter": 2e3 * np.sqrt(np.asarray(cell_area) / np.pi)}       # mm2 to µm

## 2. Where it appears

The new variable is in the declarations, and the new step in the schedule, after the parent's steps:

In [2]:
declarations = GranapAnatomyWithDiameter.declarations()
print(declarations[declarations["name"] == "equivalent_diameter"][["name", "kind", "scale", "unit"]].to_string())
print()
for stages, steps in GranapAnatomyWithDiameter.schedule():
    print(f"{', '.join(stages):>18}  {', '.join(steps)}")

                   name            kind        scale unit
25  equivalent_diameter  state_variable  Compartment   µm

  postsegmentation  _anatomies
             state  _wall_surface, _equivalent_diameter


## 3. Run it

A plant model names the subclass among its initiators (the stand-in root axis first, then the anatomy), as in
developer tutorial 1. Building the scene generates the anatomies (`initiate_plant`); each run calls the component's
steps, among them the new one:

In [3]:
import pandas as pd

from openalea.granap.metafspm.architectures import RootAxis
from openalea.granap.metafspm.component import junction_rules
from openalea.metafspm.coupling.composite_wrapper import CompositeModel
from openalea.metafspm.coupling.translator import Translator
from openalea.metafspm.scene.scene import Scene


class RootAnatomies(CompositeModel):
    initiators = (RootAxis, GranapAnatomyWithDiameter)
    from_scale, nodes, wiring = "SubOrgan", "Compartment", junction_rules

    def __init__(self, data_structure, time_step):
        self.anatomy = GranapAnatomyWithDiameter(data_structure=data_structure, time_step=time_step)
        self.declare_data_and_couple_components(translator_path=Translator(), components=(self.anatomy,))

    def run(self):
        self.anatomy()


scene = Scene(pd.DataFrame([dict(plant="plant", model=RootAnatomies, x=0., y=0., z=0., rotation=0.,
                                 scenario={"parameters": {}})]))
scene.run()
ds = scene.populations[0].data_structure
diameter = np.asarray(ds.get("equivalent_diameter"))
print(f"{ds.n_nodes()} Compartments; the largest cell: {diameter.max():.1f} µm across")

8269 Compartments; the largest cell: 64.6 µm across


## 4. Test it

A test states what the step must give, on an independent computation. In the package it would go in `test/`, run by
pytest; here it runs in place:

In [4]:
def test_equivalent_diameter_is_a_discs():
    area = np.asarray(ds.get("cell_area"))
    np.testing.assert_allclose(np.pi * (diameter / 2e3) ** 2, area, rtol=1e-12)
    assert (diameter[np.asarray(ds.get("kind")) != 0] == 0.).all()      # walls and junctions have no area


test_equivalent_diameter_is_a_discs()
print("passed")

passed


## 5. Further

- A variable another model writes is declared with `input_variable(..., by="Architecture")`, as `GranapAnatomy`'s
  `radius` and `length`. A coupling translator connects it to the providing model's variable (developer tutorial 1,
  and metafspm's tutorial).
- To make the anatomy depend on a segment's class (its diameter, its distance from the tip), the class attributes
  `root` and `shoot` take an `OrganAnatomy` with its own library and class inputs (developer tutorial 1, section 5;
  `test/component/test_class_inputs.py`).
- MECHA's components read `GranapAnatomy`'s variables in the same way, on a larger scale.